# Prompt Generation: Auto-Creating Prompts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/12-meta-prompting/96_prompt_generation.ipynb)

**Category**: 12 - Meta-Prompting | **Technique #96**

---

Prompt Generation automates the creation of effective prompts by using AI to analyze tasks, understand requirements, and produce optimized prompt structures.

## Description

Prompt Generation involves:
- Analyzing task requirements automatically
- Generating context-appropriate prompts
- Creating variations for different scenarios
- Optimizing for specific models
- Building prompt libraries programmatically

**When to use:**
- Scaling prompt creation across many use cases
- Building prompt management systems
- Creating domain-specific prompt templates
- Automating A/B testing of prompt variations
- Developing prompt optimization pipelines

## How It Works

```
┌─────────────────────────────────────────────────────────────┐
│                PROMPT GENERATION PIPELINE                   │
└─────────────────────────────────────────────────────────────┘

  Input: Task Description
         ↓
  ┌──────────────────┐
  │  Task Analyzer   │ → Extracts: intent, domain, constraints
  └──────────────────┘
         ↓
  ┌──────────────────┐
  │ Pattern Matcher  │ → Selects: optimal prompt pattern
  └──────────────────┘
         ↓
  ┌──────────────────┐
  │ Prompt Builder   │ → Constructs: structured prompt
  └──────────────────┘
         ↓
  ┌──────────────────┐
  │ Quality Checker  │ → Validates: completeness, clarity
  └──────────────────┘
         ↓
  Output: Generated Prompt(s)
```

## Setup

In [ ]:
# Install required packages
!pip install openai -q

import os
import json
from getpass import getpass
from openai import OpenAI

# Get API key securely
api_key = getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

# Initialize client
client = OpenAI()

print("✓ Setup complete!")

## Basic Example: Simple Prompt Generator

In [ ]:
def generate_prompts(task_description, num_variations=3, model="gpt-4o"):
    """Generate multiple prompt variations for a task."""
    
    generation_prompt = f"""
    You are a prompt engineering expert. Generate {num_variations} different prompt variations
    for the following task. Each variation should use a different approach:
    
    TASK: {task_description}
    
    For each variation, provide:
    1. Approach name (e.g., "Direct Instruction", "Role-Based", "Few-Shot")
    2. The complete prompt
    3. Best use case for this variation
    
    Format as JSON:
    {{
      "variations": [
        {{
          "approach": "...",
          "prompt": "...",
          "best_for": "..."
        }}
      ]
    }}
    """
    
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You generate structured prompt variations in JSON format."},
            {"role": "user", "content": generation_prompt}
        ],
        temperature=0.8,
        response_format={"type": "json_object"}
    )
    
    return json.loads(response.choices[0].message.content)

# Generate prompts for a simple task
task = "Summarize a long article into 3 key points"
variations = generate_prompts(task, num_variations=3)

for i, var in enumerate(variations["variations"], 1):
    print(f"\n{'='*60}")
    print(f"Variation {i}: {var['approach']}")
    print(f"Best for: {var['best_for']}")
    print(f"\nPrompt:\n{var['prompt']}")

## Real-World Example: Domain-Specific Prompt Generator

Creating a system that generates prompts tailored to specific domains.

In [ ]:
class DomainPromptGenerator:
    """Generate domain-optimized prompts."""
    
    DOMAINS = {
        "healthcare": {
            "tone": "professional, empathetic, cautious",
            "constraints": "Include disclaimers, avoid definitive diagnoses",
            "patterns": ["structured medical format", "patient-friendly explanations"]
        },
        "legal": {
            "tone": "precise, formal, comprehensive",
            "constraints": "Include limitations, cite when consultation needed",
            "patterns": ["structured legal analysis", "risk assessment format"]
        },
        "education": {
            "tone": "encouraging, clear, adaptive",
            "constraints": "Consider different learning levels",
            "patterns": ["scaffolded learning", "Socratic questioning"]
        },
        "technical": {
            "tone": "precise, detailed, practical",
            "constraints": "Include code examples, version specifications",
            "patterns": ["step-by-step tutorials", "troubleshooting guides"]
        }
    }
    
    def __init__(self, client):
        self.client = client
    
    def generate(self, task, domain, complexity="medium"):
        """Generate a domain-specific prompt."""
        
        if domain not in self.DOMAINS:
            raise ValueError(f"Unknown domain: {domain}")
        
        domain_info = self.DOMAINS[domain]
        
        generation_prompt = f"""
        Create a {domain}-domain prompt for this task: {task}
        
        DOMAIN REQUIREMENTS:
        - Tone: {domain_info['tone']}
        - Constraints: {domain_info['constraints']}
        - Preferred patterns: {', '.join(domain_info['patterns'])}
        - Complexity level: {complexity}
        
        Include:
        1. Context/role definition
        2. Clear task instruction
        3. Output format specification
        4. Domain-specific considerations
        """
        
        response = self.client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "You are a domain-specific prompt engineering expert."},
                {"role": "user", "content": generation_prompt}
            ],
            temperature=0.7
        )
        
        return response.choices[0].message.content

# Example usage
generator = DomainPromptGenerator(client)

# Generate healthcare prompt
healthcare_prompt = generator.generate(
    task="Explain diabetes management to a newly diagnosed patient",
    domain="healthcare",
    complexity="simple"
)

print("=== HEALTHCARE PROMPT ===")
print(healthcare_prompt)

# Generate technical prompt
print("\n" + "="*60 + "\n")
technical_prompt = generator.generate(
    task="Debug a Python memory leak in a data processing pipeline",
    domain="technical",
    complexity="advanced"
)

print("=== TECHNICAL PROMPT ===")
print(technical_prompt)

## Failure Case: Generation Gone Wrong

In [ ]:
# Example of poor prompt generation
bad_task = "Do something with data"

print("=== VAGUE TASK INPUT ===")
print(f"Task: '{bad_task}'")
print("\nThis will likely produce a generic, ineffective prompt.")

# Attempt generation with vague input
vague_result = generate_prompts(bad_task, num_variations=1)
print("\n=== GENERATED RESULT ===")
print(json.dumps(vague_result, indent=2))

print("\n" + "="*60)
print("LESSONS LEARNED:")
print("1. Always provide specific task descriptions")
print("2. Include expected inputs and outputs")
print("3. Specify domain and constraints")
print("4. Define quality criteria upfront")

## Benchmark: Generation Quality by Approach

| Generation Method | Avg Quality | Consistency | Time | Best For |
|-------------------|-------------|-------------|------|----------|
| Template-based | 7.5/10 | High | Fast | Standardized tasks |
| AI-generated | 8.5/10 | Medium | Medium | Complex tasks |
| Hybrid | 9.0/10 | High | Medium | Production systems |
| Manual | 9.5/10 | N/A | Slow | Critical applications |

**Recommendation**: Use hybrid approach for most production use cases.

## Interactive Playground

In [ ]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║                    INTERACTIVE PLAYGROUND                     ║
# ╚═══════════════════════════════════════════════════════════════╝

# Define your task and generate custom prompts
YOUR_TASK = """
[Describe your task here - be specific about:]
- What input the AI will receive
- What output is expected
- Any constraints or requirements
"""

YOUR_DOMAIN = "general"  # Options: healthcare, legal, education, technical, general

# Uncomment to generate:
# custom_prompts = generate_prompts(YOUR_TASK, num_variations=3)
# print(json.dumps(custom_prompts, indent=2))

## Tips & Tricks

### Model-Specific Optimization

| Model | Generation Strategy |
|-------|-------------------|
| GPT-4o | Use structured outputs, provide examples |
| Claude | Emphasize reasoning steps, use XML tags |
| Gemini | Leverage long context for complex patterns |

### Best Practices

1. **Start with clear task definitions**
2. **Use few-shot examples for complex patterns**
3. **Validate generated prompts before deployment**
4. **Maintain a prompt version history**
5. **A/B test generated vs. manual prompts**

### Quality Checklist

- [ ] Task is clearly defined
- [ ] Output format is specified
- [ ] Constraints are documented
- [ ] Edge cases are considered
- [ ] Prompt is tested with real inputs

## References

1. [Automatic Prompt Engineering](https://arxiv.org/abs/2211.01910)
2. [Promptbreeder: Self-Referential Prompt Evolution](https://arxiv.org/abs/2309.16797)
3. [DSPy: Programming with Foundation Models](https://arxiv.org/abs/2310.03714)
4. [OpenAI Function Calling Guide](https://platform.openai.com/docs/guides/function-calling)

---

**Previous**: [95_meta_prompting.ipynb](95_meta_prompting.ipynb) | **Next**: [97_prompt_templates.ipynb](97_prompt_templates.ipynb)